# Smart Photo Cleaner
### AI/Computer-Vision Based Duplicate & Similar Photo Detector

This notebook:
1. Scans a selected folder for images.
2. Detects exact duplicates using SHA-256.
3. Detects visually similar images using perceptual hashing (pHash).
4. Groups similar photos.
5. Scores image quality using resolution, sharpness and brightness.
6. Shows a visual review interface.
7. Lets the user select photos for recycling.
8. Moves selected files to the Windows Recycle Bin only after confirmation.

> **Safety:** The program uses `Send2Trash`, not permanent deletion.

## 1. Import libraries

In [2]:
import os
import hashlib
from collections import defaultdict

import cv2
import imagehash
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from send2trash import send2trash
from IPython.display import display, clear_output
import ipywidgets as widgets

## 2. Select the photo folder

In [3]:
# Change this path to the folder you want to scan.
photo_folder = r"D:\Realme 10 Pro +\Sshhhh\4GB Pen Drive\z.1"

if not os.path.isdir(photo_folder):
    raise FileNotFoundError(
        f"Folder not found: {photo_folder}"
    )

print("Selected folder:", photo_folder)

Selected folder: D:\Realme 10 Pro +\Sshhhh\4GB Pen Drive\z.1


## 4. Scan for images

In [4]:
IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".JPEG", ".heic"
    ".webp", ".bmp", ".gif"
)

image_files = []

for root, folders, files in os.walk(photo_folder):
    for file in files:
        if file.lower().endswith(IMAGE_EXTENSIONS):
            image_files.append(
                os.path.join(root, file)
            )

print("Total images found:", len(image_files))

Total images found: 7528


In [5]:
photo_data = []

for image_path in image_files:
    try:
        with Image.open(image_path) as img:
            photo_data.append({
                "path": image_path,
                "filename": os.path.basename(image_path),
                "width": img.width,
                "height": img.height,
                "format": img.format
            })
    except Exception as e:
        print(f"Could not read: {image_path}")
        print(f"Error: {e}")

print("Images successfully loaded:", len(photo_data))

Images successfully loaded: 7528


## 5. Detect exact duplicates with SHA-256

In [6]:
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(8192):
            sha256.update(chunk)

    return sha256.hexdigest()


hash_groups = defaultdict(list)

for photo in photo_data:
    photo["sha256"] = calculate_sha256(photo["path"])
    hash_groups[photo["sha256"]].append(photo)

exact_duplicate_groups = [
    group
    for group in hash_groups.values()
    if len(group) > 1
]

print("Exact duplicate groups:", len(exact_duplicate_groups))

for number, group in enumerate(exact_duplicate_groups, start=1):
    print(f"\nExact Duplicate Group {number}")
    for photo in group:
        print("  ", photo["filename"])

Exact duplicate groups: 0


## 6. Detect visually similar images with pHash

In [7]:
def calculate_phash(file_path):
    try:
        with Image.open(file_path) as img:
            return imagehash.phash(img)
    except Exception as e:
        print(f"Could not calculate pHash for {file_path}: {e}")
        return None


def calculate_similarity(hash1, hash2):
    if hash1 is None or hash2 is None:
        return 0.0

    distance = hash1 - hash2
    max_distance = len(hash1)

    return (1 - distance / max_distance) * 100


for photo in photo_data:
    photo["phash"] = calculate_phash(photo["path"])

print("pHash calculation completed.")

pHash calculation completed.


## 7. Find similar photo pairs

In [8]:
SIMILARITY_THRESHOLD = 90.0

similar_pairs = []

for i in range(len(photo_data)):
    for j in range(i + 1, len(photo_data)):
        photo1 = photo_data[i]
        photo2 = photo_data[j]

        similarity = calculate_similarity(
            photo1["phash"],
            photo2["phash"]
        )

        if similarity >= SIMILARITY_THRESHOLD:
            similar_pairs.append({
                "image1": photo1["filename"],
                "image2": photo2["filename"],
                "similarity": similarity,
                "path1": photo1["path"],
                "path2": photo2["path"]
            })

similar_pairs.sort(
    key=lambda x: x["similarity"],
    reverse=True
)

print("Similar pairs found:", len(similar_pairs))

for pair in similar_pairs[:10]:
    print(
        f"{pair['image1']} <--> "
        f"{pair['image2']} | "
        f"{pair['similarity']:.2f}%"
    )

Similar pairs found: 69
IMG_20250824_121505_478.jpg <--> IMG_20250824_121505_505.jpg | 100.00%
IMG_20260114_211203_318.jpg <--> IMG_20260114_211203_449.jpg | 100.00%
IMG_20250824_120720_396.jpg <--> IMG_20250824_120720_568.jpg | 96.88%
IMG_20250824_120958_769.jpg <--> IMG_20250824_120958_913.jpg | 96.88%
IMG_20251116_000231_173.jpg <--> IMG_20251116_000231_199.jpg | 96.88%
IMG_20251116_000239_771.jpg <--> IMG_20251116_000240_104.jpg | 96.88%
IMG_20251207_190338_670.jpg <--> IMG_20251210_184112_995.jpg | 96.88%
IMG_20251218_112653_784.jpg <--> IMG_20260217_202303_092.jpg | 96.88%
IMG_20260114_211138_888.jpg <--> IMG_20260114_211139_151.jpg | 96.88%
IMG_20260114_211149_434.jpg <--> IMG_20260114_211149_955.jpg | 96.88%


## 8. Group similar photos

In [9]:
def create_similarity_groups(photo_data, similar_pairs):
    parent = {
        photo["path"]: photo["path"]
        for photo in photo_data
    }

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        root_x = find(x)
        root_y = find(y)

        if root_x != root_y:
            parent[root_y] = root_x

    for pair in similar_pairs:
        union(pair["path1"], pair["path2"])

    groups = {}

    for photo in photo_data:
        root = find(photo["path"])
        groups.setdefault(root, []).append(photo)

    return [
        group
        for group in groups.values()
        if len(group) >= 2
    ]


similarity_groups = create_similarity_groups(
    photo_data,
    similar_pairs
)

print("Similar photo groups:", len(similarity_groups))

Similar photo groups: 69


## 9. Analyze image quality

In [10]:
def analyze_image_quality(file_path):
    try:
        with Image.open(file_path) as img:
            width, height = img.size

            gray_pil = img.convert("L")
            gray_array = np.array(gray_pil)
            brightness = float(np.mean(gray_array))

        image = cv2.imread(file_path)

        if image is not None:
            gray = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2GRAY
            )

            sharpness = float(
                cv2.Laplacian(
                    gray,
                    cv2.CV_64F
                ).var()
            )
        else:
            sharpness = 0.0

        return {
            "width": width,
            "height": height,
            "resolution": width * height,
            "sharpness": sharpness,
            "brightness": brightness
        }

    except Exception as e:
        print(f"Could not analyze {file_path}: {e}")

        return {
            "width": 0,
            "height": 0,
            "resolution": 0,
            "sharpness": 0.0,
            "brightness": 0.0
        }


for photo in photo_data:
    photo.update(
        analyze_image_quality(photo["path"])
    )

print("Image quality analysis completed.")

Image quality analysis completed.


In [11]:
def calculate_quality_scores(group):
    resolutions = [photo["resolution"] for photo in group]
    sharpness_values = [photo["sharpness"] for photo in group]

    min_resolution = min(resolutions)
    max_resolution = max(resolutions)

    min_sharpness = min(sharpness_values)
    max_sharpness = max(sharpness_values)

    for photo in group:

        if max_resolution == min_resolution:
            resolution_score = 1.0
        else:
            resolution_score = (
                (photo["resolution"] - min_resolution)
                / (max_resolution - min_resolution)
            )

        if max_sharpness == min_sharpness:
            sharpness_score = 1.0
        else:
            sharpness_score = (
                (photo["sharpness"] - min_sharpness)
                / (max_sharpness - min_sharpness)
            )

        brightness_score = 1 - (
            abs(photo["brightness"] - 128) / 128
        )
        brightness_score = max(0.0, brightness_score)

        photo["quality_score"] = (
            resolution_score * 0.40
            + sharpness_score * 0.45
            + brightness_score * 0.15
        )


def get_recommended_photo(group):
    return max(
        group,
        key=lambda photo: photo["quality_score"]
    )


for group in similarity_groups:
    calculate_quality_scores(group)

for number, group in enumerate(similarity_groups, start=1):
    recommended = get_recommended_photo(group)
    print(
        f"Group {number}: "
        f"Recommended -> {recommended['filename']}"
    )

Group 1: Recommended -> 20250816_100931.jpg
Group 2: Recommended -> actressfreakspot.ogs-20260310-0002.jpg
Group 3: Recommended -> gumtha_studios-02-11-2025-0064.jpg
Group 4: Recommended -> HO2JbIEaIAAQoIN.jpg
Group 5: Recommended -> IMG_20250818_190657_0455.jpg
Group 6: Recommended -> IMG_20251021_182439_371.jpg
Group 7: Recommended -> IMG_20250824_120720_396.jpg
Group 8: Recommended -> IMG_20250824_120723_125.jpg
Group 9: Recommended -> IMG_20250824_120817_451.jpg
Group 10: Recommended -> IMG_20250824_120828_347.jpg
Group 11: Recommended -> IMG_20250824_120918_604.jpg
Group 12: Recommended -> IMG_20250824_120927_883.jpg
Group 13: Recommended -> IMG_20260121_221922_176.jpg
Group 14: Recommended -> IMG_20250824_120942_854.jpg
Group 15: Recommended -> IMG_20250824_120952_370.jpg
Group 16: Recommended -> IMG_20250824_120958_913.jpg
Group 17: Recommended -> IMG_20250824_121328_0050.jpg
Group 18: Recommended -> IMG_20250824_121344_995.jpg
Group 19: Recommended -> IMG_20250824_121505_505.jp

## 10. Safe interactive review and Recycle Bin

In [12]:
def move_to_recycle_bin(file_path):
    try:
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            return False

        send2trash(file_path)
        return True

    except Exception as e:
        print(f"Could not recycle {file_path}: {e}")
        return False


class SafePhotoReviewer:

    def __init__(self, groups):
        self.groups = groups
        self.current_group = 0
        self.selected_for_recycle = []
        self.checkboxes = []
        self.output = widgets.Output()

        self.show_group()

    def show_group(self):
        with self.output:
            clear_output(wait=True)

            if self.current_group >= len(self.groups):
                self.show_final_summary()
                return

            group = self.groups[self.current_group]
            recommended = get_recommended_photo(group)

            print("=" * 70)
            print(
                f"SIMILAR PHOTO GROUP "
                f"{self.current_group + 1} / {len(self.groups)}"
            )
            print("=" * 70)
            print(
                f"⭐ Recommended to Keep: "
                f"{recommended['filename']}"
            )
            print(
                f"Quality Score: "
                f"{recommended['quality_score']:.3f}"
            )
            print()

            number_of_images = len(group)

            plt.figure(
                figsize=(5 * number_of_images, 6)
            )

            for i, photo in enumerate(group):
                with Image.open(photo["path"]) as img:
                    image = img.copy()

                plt.subplot(
                    1,
                    number_of_images,
                    i + 1
                )
                plt.imshow(image)

                label = (
                    "⭐ RECOMMENDED\n"
                    if photo["path"] == recommended["path"]
                    else ""
                )

                plt.title(
                    f"{label}{photo['filename']}\n"
                    f"{photo['width']} × {photo['height']}\n"
                    f"Sharpness: {photo['sharpness']:.1f}\n"
                    f"Score: {photo['quality_score']:.3f}"
                )
                plt.axis("off")

            plt.tight_layout()
            plt.show()

            print(
                "Select the photos you want "
                "to move to the Recycle Bin:"
            )
            print()

            self.checkboxes = []

            for photo in group:
                checkbox = widgets.Checkbox(
                    value=False,
                    description=f"Recycle: {photo['filename']}"
                )

                checkbox.photo_path = photo["path"]
                checkbox.is_recommended = (
                    photo["path"] == recommended["path"]
                )

                self.checkboxes.append(checkbox)
                display(checkbox)

            print()

            next_button = widgets.Button(
                description="Next Group",
                button_style="info"
            )
            skip_button = widgets.Button(
                description="Skip Group"
            )

            next_button.on_click(self.next_group)
            skip_button.on_click(self.skip_group)

            display(
                widgets.HBox([
                    next_button,
                    skip_button
                ])
            )

    def next_group(self, button):
        selected = []

        for checkbox in self.checkboxes:
            if checkbox.value:
                selected.append({
                    "path": checkbox.photo_path,
                    "is_recommended": checkbox.is_recommended
                })

        if not selected:
            self.current_group += 1
            self.show_group()
            return

        if any(item["is_recommended"] for item in selected):
            self.show_recommended_warning(selected)
        else:
            self.add_selected_files(selected)

    def show_recommended_warning(self, selected):
        with self.output:
            clear_output(wait=True)

            print("=" * 70)
            print("⚠️ WARNING")
            print("=" * 70)
            print()
            print(
                "You selected the photo that the system "
                "recommends keeping."
            )
            print()
            print(
                "Are you sure you want to move it "
                "to the Recycle Bin?"
            )
            print()

            confirm_button = widgets.Button(
                description="Yes, Continue",
                button_style="danger"
            )
            cancel_button = widgets.Button(
                description="Cancel"
            )

            confirm_button.on_click(
                lambda b: self.add_selected_files(selected)
            )
            cancel_button.on_click(
                lambda b: self.show_group()
            )

            display(
                widgets.HBox([
                    confirm_button,
                    cancel_button
                ])
            )

    def add_selected_files(self, selected):
        for item in selected:
            if item["path"] not in self.selected_for_recycle:
                self.selected_for_recycle.append(item["path"])

        self.current_group += 1
        self.show_group()

    def skip_group(self, button):
        self.current_group += 1
        self.show_group()

    def show_final_summary(self):
        with self.output:
            clear_output(wait=True)

            count = len(self.selected_for_recycle)

            print("=" * 70)
            print("🎉 REVIEW COMPLETED")
            print("=" * 70)
            print()
            print(
                f"Photos selected for recycling: {count}"
            )
            print()

            if count == 0:
                print("No photos were selected.")
                return

            for i, path in enumerate(
                self.selected_for_recycle,
                start=1
            ):
                print(f"{i}. {path}")

            print()

            recycle_button = widgets.Button(
                description=(
                    f"Move {count} Photo(s) to Recycle Bin"
                ),
                button_style="danger"
            )
            cancel_button = widgets.Button(
                description="Cancel"
            )

            recycle_button.on_click(
                self.final_recycle_confirmation
            )
            cancel_button.on_click(
                self.cancel_recycle
            )

            display(
                widgets.HBox([
                    recycle_button,
                    cancel_button
                ])
            )

    def final_recycle_confirmation(self, button):
        with self.output:
            clear_output(wait=True)

            count = len(self.selected_for_recycle)

            print("=" * 70)
            print("⚠️ FINAL CONFIRMATION")
            print("=" * 70)
            print()
            print(
                f"You are about to move {count} photo(s) "
                "to the Windows Recycle Bin."
            )
            print()
            print(
                "The files will not be permanently deleted."
            )
            print()

            confirm_button = widgets.Button(
                description="♻️ Yes, Move to Recycle Bin",
                button_style="danger"
            )
            cancel_button = widgets.Button(
                description="Cancel"
            )

            confirm_button.on_click(
                self.perform_recycle
            )
            cancel_button.on_click(
                self.cancel_recycle
            )

            display(
                widgets.HBox([
                    confirm_button,
                    cancel_button
                ])
            )

    def perform_recycle(self, button):
        with self.output:
            clear_output(wait=True)

            print("=" * 70)
            print("♻️ MOVING PHOTOS TO RECYCLE BIN")
            print("=" * 70)

            successful = 0
            failed = 0

            for path in self.selected_for_recycle:
                if move_to_recycle_bin(path):
                    successful += 1
                    print(f"✓ Recycled: {path}")
                else:
                    failed += 1
                    print(f"✗ Failed: {path}")

            print()
            print(f"Successfully recycled: {successful}")
            print(f"Failed: {failed}")
            print()
            print(
                "Check the Windows Recycle Bin if you need "
                "to restore a file."
            )

    def cancel_recycle(self, button):
        with self.output:
            clear_output(wait=True)
            print("❌ Recycle operation cancelled.")
            print("No files were moved.")


reviewer = SafePhotoReviewer(similarity_groups)
display(reviewer.output)

Output()